# [STARTER] Udaplay Project

## Part 02 - Agent

In this part of the project, you'll use your VectorDB to be part of your Agent as a tool.

You're building UdaPlay, an AI Research Agent for the video game industry. The agent will:
1. Answer questions using internal knowledge (RAG)
2. Search the web when needed
3. Maintain conversation state
4. Return structured outputs
5. Store useful information for future use

### Setup

In [46]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [59]:
# TODO: Import the necessary libs
# For example: 
import os
import chromadb

from dotenv import load_dotenv
from lib.agents import Agent
from lib.llm import LLM
from lib.messages import UserMessage, SystemMessage, ToolMessage, AIMessage
from lib.tooling import tool
from lib.state_machine import StateMachine, Step, EntryPoint, Transition, Termination
from lib.tooling import Tool, ToolCall
from pydantic import BaseModel, Field
from typing import List, Optional, TypedDict, NotRequired, Literal
from tavily import TavilyClient


In [48]:
# TODO: Load environment variables
load_dotenv(override=True)

# OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
# TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

True

In [49]:
class RetrievalDocument(BaseModel):
    Platform: str = Field(description="The platform of the document")
    Name: str = Field(description="The name of the document")
    YearOfRelease: int = Field(description="The year the document was released")
    Description: str = Field(description="description of the document")

### Tools

Build at least 3 tools:
- retrieve_game: To search the vector DB
- evaluate_retrieval: To assess the retrieval performance
- game_web_search: If no good, search the web


#### Retrieve Game Tool

In [61]:
# TODO: Create retrieve_game tool
# It should use chroma client and collection you created
# chroma_client = chromadb.PersistentClient(path="chromadb")
# collection = chroma_client.get_collection("udaplay")
# Tool Docstring:
#    Semantic search: Finds most results in the vector DB
#    args:
#    - query: a question about game industry. 
#
#    You'll receive results as list. Each element contains:
#    - Platform: like Game Boy, Playstation 5, Xbox 360...)
#    - Name: Name of the Game
#    - YearOfRelease: Year when that game was released for that platform
#    - Description: Additional details about the game

def retrieve_game(query: str) -> list[RetrievalDocument]:
    """
    Semantic search: Finds most results in the vector DB
    args:
    - query: a question about game industry. 
    """
    chroma_client = chromadb.PersistentClient(path="chromadb")
    collection = chroma_client.get_collection("udaplay")
    query_result = collection.query(query_texts=[query], n_results=3)
    results = []
    for item in query_result["metadatas"][0]:
        results.append(RetrievalDocument(                                                        
            Platform=item["Platform"],                                                           
            Name=item["Name"],                                                                   
            YearOfRelease=item["YearOfRelease"],                                                 
            Description=item["Description"]
        ))    
    return results
                                                 

#### Evaluate Retrieval Tool

In [51]:
# TODO: Create evaluate_retrieval tool
# You might use an LLM as judge in this tool to evaluate the performance
# You need to prompt that LLM with something like:
# "Your task is to evaluate if the documents are enough to respond the query. "
# "Give a detailed explanation, so it's possible to take an action to accept it or not."
# Use EvaluationReport to parse the result
# Tool Docstring:
#    Based on the user's question and on the list of retrieved documents, 
#    it will analyze the usability of the documents to respond to that question. 
#    args: 
#    - question: original question from user
#    - retrieved_docs: retrieved documents most similar to the user query in the Vector Database
#    The result includes:
#    - useful: whether the documents are useful to answer the question
#    - description: description about the evaluation result
class EvaluationReport(BaseModel):
    useful: bool = Field(description="whether the documents are useful to answer the question")
    description: str = Field(description="description about the evaluation result")

def evaluate_retrieval(question: str, retrieved_docs: list[RetrievalDocument]) -> EvaluationReport:
    if not retrieved_docs:
        return EvaluationReport(useful=False, description="No documents were retrieved.")
    prompt = f"""
    Your task is to judge if the documents are relevant/complete to respond the query. 
    Give a detailed explanation, so it's possible to take an action to accept it or not.
    
    Question: {question}
    
    Retrieved Documents:
    """
    for doc in retrieved_docs:
        prompt += f"""
        Platform: {doc.Platform}
        Name: {doc.Name}
        YearOfRelease: {doc.YearOfRelease}
        Description: {doc.Description}
        """

    llm = LLM(api_key=os.getenv("OPENAI_API_KEY"))
    response = llm.invoke(input=prompt, response_format=EvaluationReport)
    return EvaluationReport.model_validate_json(response.content)
    

    

#### Game Web Search Tool

In [52]:
# TODO: Create game_web_search tool
# Please use Tavily client to search the web
# Tool Docstring:
#    Semantic search: Finds most results in the vector DB
#    args:
#    - question: a question about game industry. 

def game_web_search(question: str) -> str:
    """
    search the web using Tavily client to find an answer for the question.
    args:
        - question: a question about game industry. 
    return the answer from the web search.
    """
    client = TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))
    response = client.search(query=question)
    if isinstance(response, dict) and response.get("answer"):
        return response["answer"]
    return "No answer found from web search."

### Agent

In [53]:
# TODO: Create your Agent abstraction using StateMachine
# Equip with an appropriate model
# Craft a good set of instructions 
# Plug all Tools you developed

class AgentState(TypedDict):
    question: str
    retrieved_docs: NotRequired[list[RetrievalDocument]]
    evaluation: NotRequired[EvaluationReport]
    source: NotRequired[Literal["local", "web"]]
    web_answer: NotRequired[str | None]
    final_answer: NotRequired[str | None]

def process_query_fn(state: AgentState) -> dict:
    # Process the user query and decide which tool to call
    # This is a placeholder implementation. You should implement the logic to decide which tool to call based on the user query and the retrieved documents.
    docs = retrieve_game(state["question"])
    evaluation = evaluate_retrieval(state["question"], docs)
    if evaluation.useful:
        return {
            "retrieved_docs": docs,
            "evaluation": evaluation,
            "source": "local",
        }
    web_answer = game_web_search(state["question"])
    return {
        "retrieved_docs": docs,
        "evaluation": evaluation,
        "web_answer": web_answer,
        "source": "web",
    }

def generate_result_fn(state: AgentState) -> dict:
    # Generate the final answer based on the tool results and the conversation history
    # This is a placeholder implementation. You should implement the logic to generate a response based on the tool results and the conversation history.
    if state.get("source") == "local":
        answer = f"Based on the retrieved documents, here is the information I found:\n"
        for doc in state["retrieved_docs"]:
            answer += f"- {doc.Name} on {doc.Platform} released in {doc.YearOfRelease}: {doc.Description}\n"
        return {"final_answer": answer}
    elif state.get("source") == "web":
        return {"final_answer": f"Based on the web search, here is the information I found:\n{state.get('web_answer')}"}
    else:
        return {"final_answer": "I'm sorry, I couldn't find relevant information to answer your question."}

entry = EntryPoint[AgentState]()
process_query = Step[AgentState]("process_query",process_query_fn)
generate_result = Step[AgentState]("generate_result",generate_result_fn)
termination = Termination[AgentState]()
workflow = StateMachine[AgentState](AgentState)

workflow.add_steps([entry, process_query, generate_result, termination])

workflow.connect(entry, process_query)
workflow.connect(process_query, generate_result)
workflow.connect(generate_result, termination)


In [ ]:
# TODO: Invoke your agent
# - When Pokémon Gold and Silver was released?
# - Which one was the first 3D platformer Mario game?
# - Was Mortal Kombat X realeased for Playstation 5?

questions = [
    "When Pokémon Gold and Silver was released?",
    "Which one was the first 3D platformer Mario game",
    "Was Mortal Kombat X realeased for Playstation 5"
]

for q in questions:
    initial_state: AgentState = {"question": q}
    result = workflow.run(initial_state)   # or invoke/execute depending on API
    final_state = result.get_final_state()
    print("=" * 60)
    print("Question:", q)
    print(final_state["final_answer"])

[StateMachine] Starting: __entry__
[StateMachine] Executing step: process_query
[StateMachine] Executing step: generate_result
[StateMachine] Terminating: __termination__
Question: When Pokémon Gold and Silver was released?
Based on the retrieved documents, here is the information I found:
- Pokémon Gold and Silver on Game Boy Color released in 1999: Second-generation Pokémon games introducing new regions, Pokémon, and gameplay mechanics.
- Pokémon Ruby and Sapphire on Game Boy Advance released in 2002: Third-generation Pokémon games set in the Hoenn region, featuring new Pokémon and double battles.
- Super Mario 64 on Nintendo 64 released in 1996: A groundbreaking 3D platformer that set new standards for the genre, featuring Mario's quest to rescue Princess Peach.

[StateMachine] Starting: __entry__
[StateMachine] Executing step: process_query
[StateMachine] Executing step: generate_result
[StateMachine] Terminating: __termination__
Question: Which one was the first 3D platformer Mari

### (Optional) Advanced

In [ ]:
# TODO: Update your agent with long-term memory
# TODO: Convert the agent to be a state machine, with the tools being pre-defined nodes